# 👥 MathAI — Gerenciador de Usuários, Sessões & Perfis Cognitivos

Caderno interativo para administração, consulta e diagnóstico de contas de estudantes, segurança (2FA/Sessões) e métricas de engajamento no banco de dados (**SQLite Local** ou **Turso Cloud**).

---

## ⚙️ 1. Inicialização e Conexão com o Banco
Configura o caminho do projeto e conecta automaticamente à base de dados ativa (`.env`).

In [ ]:
import sys
import os
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, HTML

# Adiciona a raiz do projeto no sys.path
ROOT_DIR = Path.cwd().parent.parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Limpa qualquer cache em memória do kernel
for mod in ["src", "src.database", "src.database.db", "db_supabase"]:
    sys.modules.pop(mod, None)

from src.database.db import pegar_conexao, obter_status_banco, get_supabase

# Conexão direta com Supabase
supabase = get_supabase()
con = pegar_conexao()
origem = obter_status_banco()

print("✅ Conexão estabelecida com sucesso!")
print(f"📊 Banco de Dados Ativo: {origem}")

## 📊 2. Visão Geral e Indicadores Rápidos (KPIs)
Métricas gerais sobre o volume de usuários, status de verificação (2FA) e distribuição de estudos.

In [ ]:
con = pegar_conexao()

# Total de Usuários e Status 2FA
df_total = pd.read_sql("""
    SELECT 
        COUNT(*) AS total_usuarios,
        SUM(CASE WHEN verificado = 1 THEN 1 ELSE 0 END) AS verificados,
        SUM(CASE WHEN verificado = 0 THEN 1 ELSE 0 END) AS pendentes_2fa
    FROM usuarios;
""", con)

# Total de Sessões Lembradas
df_sessoes = pd.read_sql("SELECT COUNT(*) AS total_sessoes_ativas FROM sessoes_lembradas;", con)

# Total de Tentativas Registradas
df_tentativas = pd.read_sql("SELECT COUNT(*) AS total_tentativas FROM tentativas;", con)

print("=" * 60)
print("  📊 RESUMO GERAL DO SISTEMA")
print("=" * 60)
print(f"👥 Total de Contas:        {df_total['total_usuarios'].iloc[0]}")
print(f"🔒 Contas Verificadas (2FA): {df_total['verificados'].iloc[0]}")
print(f"⏳ Contas Pendentes:        {df_total['pendentes_2fa'].iloc[0]}")
print(f"🔑 Sessões Ativas (Tokens):  {df_sessoes['total_sessoes_ativas'].iloc[0]}")
print(f"🎯 Tentativas de Questões:   {df_tentativas['total_tentativas'].iloc[0]}")
print("=" * 60)

## 📋 3. Tabela Completa de Usuários Cadastrados
Visualização tabular com todos os dados de perfil, escolaridade, faculdade, localização e contato.

In [ ]:
con = pegar_conexao()
df_usuarios = pd.read_sql("""
    SELECT 
        id, 
        nome, 
        email, 
        celular, 
        cidade, 
        estado, 
        escolaridade, 
        faculdade, 
        curso, 
        motivos,
        concursos_foco,
        verificado, 
        criado_em
    FROM usuarios
    ORDER BY id ASC;
""", con)

# Formata coluna de verificação para visualização amigável
df_usuarios['status_2fa'] = df_usuarios['verificado'].apply(lambda x: '✅ Ativo' if x == 1 else '⏳ Pendente')

display(df_usuarios[['id', 'nome','email', 'celular', 'cidade', 'estado', 'faculdade', 'curso', 'status_2fa', 'criado_em']])

## 🔍 4. Relatório Formatado no Console (`ver_usuarios`)
Gera o relatório em formato texto/console identado (equivalente ao utilitário `ver_usuarios.py`).

In [ ]:
def listar_usuarios_formatado():
    con = pegar_conexao()
    cur = con.cursor()
    cur.execute("""
        SELECT id, nome, email, celular, cpf, cidade, estado, escolaridade, faculdade, curso, verificado, criado_em
        FROM usuarios
        ORDER BY id ASC
    """)
    usuarios = cur.fetchall()
    
    if not usuarios:
        print("[i] Nenhum usuário cadastrado no momento.")
        return

    print("\n" + "=" * 118)
    print(f"  MATHAI - USUÁRIOS CADASTRADOS (TOTAL: {len(usuarios)})  [Banco: {origem}]")
    print("=" * 118)
    print(f"{'ID':<4} | {'NOME':<25} | {'E-MAIL':<28} | {'CELULAR':<15} | {'CIDADE/UF':<12} | {'CRIADO EM':<16} | {'2FA'}")
    print("-" * 118)

    for u in usuarios:
        cid_uf = f"{u['cidade'] or '-'}/{u['estado'] or '-'}"
        if len(cid_uf) > 12:
            cid_uf = cid_uf[:10] + ".."
        status_2fa = "Ativo" if u["verificado"] == 1 else "Pendente"
        nome = (u["nome"] or "-")[:24]
        email = (u["email"] or "-")[:27]
        cel = (u["celular"] or "-")[:14]
        criado = (u["criado_em"] or "-")[:16]

        print(f"#{u['id']:<3} | {nome:<25} | {email:<28} | {cel:<15} | {cid_uf:<12} | {criado:<16} | {status_2fa}")

    print("=" * 118 + "\n")

listar_usuarios_formatado()

## 🕵️ 5. Ficha Individual de Estudante (Diagnóstico Completo)
Insira o ID ou E-mail do aluno para ver todos os detalhes cadastrais, histórico de resoluções e interações com IA.

In [ ]:
def ficha_do_aluno(identificador):
    """
    Busca a ficha completa de um aluno por ID (int) ou Email (str).
    """
    con = pegar_conexao()
    cur = con.cursor()
    
    if isinstance(identificador, int) or str(identificador).isdigit():
        cur.execute("SELECT * FROM usuarios WHERE id = ?", (int(identificador),))
    else:
        cur.execute("SELECT * FROM usuarios WHERE email = ?", (str(identificador).strip().lower(),))
        
    aluno = cur.fetchone()
    if not aluno:
        print(f"❌ Aluno '{identificador}' não encontrado.")
        return
        
    aid = aluno['id']
    print("=" * 80)
    print(f"👤 FICHA DO ALUNO: #{aid} - {aluno['nome']} ({aluno['email']})")
    print("=" * 80)
    print(f"📞 Celular:      {aluno['celular'] or 'Não informado'}")
    print(f"📍 Localização:  {aluno['cidade'] or '-'}/{aluno['estado'] or '-'} (CEP: {aluno['cep'] or '-'})")
    print(f"🎓 Escolaridade: {aluno['escolaridade'] or '-'} | Faculdade: {aluno['faculdade'] or '-'} | Curso: {aluno['curso'] or '-'}")
    print(f"🎯 Concursos:    {aluno['concursos_foco'] or '-'}")
    print(f"💡 Motivos:      {aluno['motivos'] or '-'}")
    print(f"🔒 2FA:          {'✅ Verificado' if aluno['verificado'] == 1 else '⏳ Pendente'}")
    print(f"📅 Cadastro em:  {aluno['criado_em']}")
    print("-" * 80)
    
    # Histórico de Tentativas
    df_tent = pd.read_sql("""
        SELECT 
            t.id AS tentativa_id,
            q.materia,
            q.topico,
            q.banca,
            q.ano,
            t.acertou,
            t.tempo_segundos,
            t.tipo_erro,
            t.estrategia_usada,
            t.data_hora
        FROM tentativas t
        JOIN questoes q ON q.id = t.questao_id
        WHERE t.aluno_id = ?
        ORDER BY t.data_hora DESC
    """, con, params=(aid,))
    
    print(f"📝 Histórico de Resoluções ({len(df_tent)} tentativas):")
    if len(df_tent) > 0:
        display(df_tent.head(10))
    else:
        print("   Nenhuma questão resolvida até o momento.")
        
    # Revisões Agendadas (SM-2)
    df_sm2 = pd.read_sql("""
        SELECT item_tipo, item_id, fator_facilidade, intervalo_dias, repeticoes, proxima_revisao
        FROM revisao_espacada
        WHERE aluno_id = ?
        ORDER BY proxima_revisao ASC
    """, con, params=(aid,))
    
    if len(df_sm2) > 0:
        print(f"\n⏰ Próximas Revisões Espaçadas (SM-2):")
        display(df_sm2.head(5))
    print("=" * 80)

# Exemplo de uso: substitua pelo ID ou email desejado
# ficha_do_aluno(1)
if len(df_usuarios) > 0:
    primeiro_id = int(df_usuarios['id'].iloc[0])
    ficha_do_aluno(primeiro_id)

## 🔑 6. Gerenciamento de 2FA & Códigos de Verificação
Consulte os últimos códigos OTP gerados e forneça suporte a estudantes que precisem de ativação manual.

In [ ]:
con = pegar_conexao()
df_codigos = pd.read_sql("""
    SELECT id, email, codigo, expira_em, usado, criado_em
    FROM codigos_verificacao
    ORDER BY criado_em DESC
    LIMIT 15;
""", con)

print("🔐 Últimos 15 Códigos OTP / 2FA Gerados:")
display(df_codigos)

### ⚡ Ativar Conta Manualmente (Suporte Administrativo)
Caso um usuário tenha problemas para receber o código por e-mail, descomente e execute a função abaixo para validá-lo diretamente:

In [ ]:
def ativar_usuario_manualmente(email_ou_id):
    con = pegar_conexao()
    cur = con.cursor()
    if isinstance(email_ou_id, int) or str(email_ou_id).isdigit():
        cur.execute("UPDATE usuarios SET verificado = 1 WHERE id = ?", (int(email_ou_id),))
    else:
        cur.execute("UPDATE usuarios SET verificado = 1 WHERE email = ?", (str(email_ou_id).strip().lower(),))
    con.commit()
    print(f"✅ Usuário '{email_ou_id}' ativado com sucesso!")

# Exemplo: ativar_usuario_manualmente('aluno@example.invalid')

## 🚪 7. Sessões Lembradas & Tokens de Login
Permite monitorar dispositivos conectados e limpar sessões expiradas.

In [ ]:
con = pegar_conexao()
df_sessoes_ativas = pd.read_sql("""
    SELECT 
        s.token,
        s.usuario_id,
        u.nome,
        u.email,
        s.expira_em,
        s.criado_em
    FROM sessoes_lembradas s
    JOIN usuarios u ON u.id = s.usuario_id
    ORDER BY s.criado_em DESC;
""", con)

print(f"🔑 Total de Sessões Ativas: {len(df_sessoes_ativas)}")
display(df_sessoes_ativas.head(10))

In [ ]:
def limpar_sessoes_expiradas():
    """
    Remove sessões cujo prazo de validade já expirou.
    """
    con = pegar_conexao()
    cur = con.cursor()
    cur.execute("DELETE FROM sessoes_lembradas WHERE expira_em < CURRENT_TIMESTAMP;")
    con.commit()
    print("🧹 Sessões expiradas limpas com sucesso!")

# Descomente para executar a limpeza:
# limpar_sessoes_expiradas()

## 📈 8. Distribuição Demográfica e Interesses
Gráficos e tabelas para entender quem são os alunos (bancas de foco, faculdades e motivos de estudo).

In [ ]:
con = pegar_conexao()
df_esc = pd.read_sql("SELECT escolaridade, COUNT(*) AS quantidade FROM usuarios GROUP BY escolaridade ORDER BY quantidade DESC;", con)
df_fac = pd.read_sql("SELECT faculdade, COUNT(*) AS quantidade FROM usuarios WHERE faculdade IS NOT NULL GROUP BY faculdade ORDER BY quantidade DESC;", con)

print("🎓 Distribuição por Escolaridade:")
display(df_esc)

print("🏛️ Distribuição por Faculdade / Instituição:")
display(df_fac.head(10))